 # Day 6 (Thu Aug 20) — Let's build the GPT Tokenizer (2h13m)

He types everything live, so this is a scratchpad, not a starter. His filled colab + minbpe repo are answer keys — stay out of them until it's written.

**why this day matters most:** 28 of the 48 CS336 tests are tokenizer tests (`test_train_bpe.py` 3 + `test_tokenizer.py` 25). It also unblocks the TinyStories run — nothing trains until my tokenizer can encode the corpus.

**arc (video chapters):**
- 14:56 unicode code points · 18:15 UTF-8 and why bytes
- 23:50 BPE algorithm · 27:02 implementation starts
- 28:35 get_stats · 30:36 merge · 34:58 the training loop + compression ratio
- 42:47 decode · 48:21 encode
- 57:36 **regex splitting** — the part that makes it a real tokenizer
- 1:11:38 tiktoken, GPT-2 vs GPT-4 patterns · 1:18:26 special tokens
- 1:25:28 exercise time (build your own GPT-4 tokenizer)
- 1:51:41 the quirks: why models can't spell, 9.11 vs 9.9, SolidGoldMagikarp

**targets:**
1. `train(text, vocab_size)` → vocab + merges. CS336 wants 500-token vocab on `corpus.en` in **under 1.5 seconds** — naive O(n²) passes correctness and fails the clock
2. `encode` / `decode` round-trip on unicode, and matching tiktoken exactly
3. special tokens that never get merged into
4. consolidate into `bpe.py`, wire into `05-cs336/assignment1-basics/tests/adapters.py`

**rule:** watch a chapter → close it → write it here → only then compare to the answer key.

**sample text is in `data/`, no inline pasting:**
- `unicode_intro.txt` — Reed Beta's "Programmer's Intro to Unicode", 23,394 chars / 24,652 utf-8 bytes / 213 distinct codepoints. The training corpus (same source karpathy uses)
- `unicode_torture.txt` — fullwidth, circled letters, flag emoji, interrobang. Round-trip must survive this
- `hello_korean.txt` — 26 chars, 39 bytes: the one-liner that shows chars != bytes

bigger corpora already on disk: `03-gpt/input.txt` (1.1MB shakespeare) and `05-cs336/assignment1-basics/tests/fixtures/` (corpus.en for the 1.5s speed test, tinystories_sample.txt)

In [1]:
import regex as re
from collections import Counter

In [ ]:
from pathlib import Path

D = Path("data")
text = (D / "unicode_intro.txt").read_text()          # 23k chars, 213 distinct codepoints — the training corpus
torture = (D / "unicode_torture.txt").read_text()     # fullwidth, circled, flag emoji, interrobang
korean = (D / "hello_korean.txt").read_text()

print(len(text), "chars |", len(text.encode("utf-8")), "utf-8 bytes")
print(torture)

## scratch

In [ ]:
# Hyperparams


In [ ]:
with open('input.txt', 'r') as f:
    text = f.read()
print(len(text))
print(text[:200])